# Параметрический анализ численности маргариток

В данном скрипте выполняется систематическое исследование влияния
различных параметров на динамику численности чёрных и белых маргариток
в модели Daisyworld. Для каждого набора параметров строится график
изменения количества маргариток во времени.

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"

using Agents
using DataFrames
using Plots
using CairoMakie

### Подключение модели

Импортируем определение модели Daisyworld из исходного файла.

In [ ]:
include(srcdir("daisyworld.jl"))

## Определение агрегатных функций

Для сбора статистики о популяции маргариток определим две функции,
которые проверяют принадлежность агента к определённому виду:
- `black(a)` — возвращает `true`, если маргаритка чёрная
- `white(a)` — возвращает `true`, если маргаритка белая

In [ ]:
black(a) = a.breed == :black
white(a) = a.breed == :white

### Агрегатные данные

`adata` определяет, какие данные о агентах будут собираться в процессе
моделирования. Здесь мы собираем количество чёрных и белых маргариток
на каждом шаге.

In [ ]:
adata = [(black, count), (white, count)]

## Определение параметров эксперимента

### Структура параметров

Для исследования задаётся словарь параметров, где некоторые параметры
представлены в виде векторов. Это позволяет автоматически генерировать
все возможные комбинации значений.

**Исследуемые параметры:**
- `max_age` — максимальный возраст маргариток (25 и 40)
- `init_white` — начальная доля белых маргариток (0.2 и 0.8)

**Фиксированные параметры:**
- `griddims` — размер сетки (30×30)
- `init_black` — начальная доля чёрных маргариток (0.2)
- `albedo_white` — альбедо белых маргариток (0.75)
- `albedo_black` — альбедо чёрных маргариток (0.25)
- `surface_albedo` — альбедо почвы (0.4)
- `solar_change` — скорость изменения светимости (0.005)
- `solar_luminosity` — начальная светимость (1.0)
- `scenario` — сценарий изменения светимости (:default)
- `seed` — начальное значение для генератора случайных чисел (165)

In [ ]:
param_dict = Dict(
    :griddims => (30, 30),
    :max_age => [25, 40],
    :init_white => [0.2, 0.8],
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :default,
    :seed => 165,
)

## Генерация комбинаций параметров

Функция `dict_list` из пакета DrWatson создаёт все возможные комбинации
параметров из заданного словаря. Для каждого параметра, представленного
вектором, генерируются отдельные эксперименты.

In [ ]:
params_list = dict_list(param_dict)

## Цикл по всем комбинациям параметров

Для каждого набора параметров выполняется:
1. Создание модели с заданными параметрами
2. Запуск симуляции на 1000 шагов сбора данных
3. Построение графика численности маргариток
4. Сохранение графика с уникальным именем

In [ ]:
for params in params_list

### Создание модели

Модель инициализируется с текущим набором параметров.
Используется синтаксис `;params...` для распаковки словаря
в именованные аргументы.

In [ ]:
    model = daisyworld(; params...)

### Запуск моделирования

Запускаем симуляцию на 1000 шагов. Результаты сохраняются в DataFrame:
- `agent_df` — данные об агентах (количество чёрных и белых маргариток)
- `model_df` — данные о модели (не используются в данном скрипте)

In [ ]:
    agent_df, model_df = run!(model, 1000; adata)

### Построение графика численности

#### Создание фигуры и осей

Создаём фигуру размером 600×400 пикселей с одной осью,
где по оси X откладывается время (tick), а по оси Y — количество маргариток.

In [ ]:
    figure = Figure(size = (600, 400))
    ax = figure[1, 1] = Axis(figure,
        xlabel = "tick",
        ylabel = "daisy count"
    )

#### Отображение динамики чёрных маргариток

Строим линию для чёрных маргариток. Цвет линии — чёрный.

In [ ]:
    blackl = lines!(ax,
        agent_df[!, :time],
        agent_df[!, :count_black],
        color = :black
    )

#### Отображение динамики белых маргариток

Строим линию для белых маргариток. Цвет линии — оранжевый.

In [ ]:
    whitel = lines!(ax,
        agent_df[!, :time],
        agent_df[!, :count_white],
        color = :orange
    )

#### Добавление легенды

Добавляем легенду, которая идентифицирует линии на графике.
Легенда размещается справа от основного графика.

In [ ]:
    Legend(figure[1, 2],
        [blackl, whitel],
        ["black", "white"],
        labelsize = 12
    )

### Формирование имени файла

Используем функцию `savename` из пакета DrWatson для автоматического
формирования уникального имени файла на основе параметров эксперимента.
Это обеспечивает воспроизводимость и удобство идентификации результатов.

In [ ]:
    plt_name = savename("daisy-count", params) * ".png"

### Сохранение результата

Сохраняем полученный график в каталог `plots/` с соответствующим именем.

In [ ]:
    save(plotsdir(plt_name), figure)
end

## Интерпретация результатов

После выполнения скрипта в каталоге `plots/` появятся графики для
каждой комбинации параметров. Анализ этих графиков позволяет:

1. **Исследовать влияние максимального возраста маргариток**:
   - При `max_age = 25` наблюдается более быстрая смена поколений,
     что может приводить к более резким колебаниям численности
   - При `max_age = 40` динамика более сглажена, популяции стабильнее

2. **Исследовать влияние начального соотношения видов**:
   - При `init_white = 0.2` изначально преобладают чёрные маргаритки,
     что приводит к более высокой температуре и изменению условий
     для размножения белых маргариток
   - При `init_white = 0.8` изначально преобладают белые маргаритки,
     что приводит к более низкой температуре и преимуществу для
     чёрных маргариток

3. **Анализ установления равновесия**:
   - На начальном этапе (первые 100-200 шагов) наблюдается переходный
     процесс, связанный с адаптацией системы к начальным условиям
   - После переходного периода система приходит к стабильному состоянию,
     где численность обоих видов выходит на постоянный уровень

4. **Оценка устойчивости системы**:
   - Несмотря на разные начальные условия, финальное равновесное
     состояние оказывается одинаковым для всех экспериментов
   - Это демонстрирует свойство аттрактора — система стремится
     к одному и тому же состоянию независимо от начальных условий

### Важное замечание

Поскольку используется фиксированный `seed` (165), все эксперименты
являются детерминированными и воспроизводимыми. При необходимости
исследования стохастической природы модели можно варьировать
параметр `seed` или отключить его фиксацию.